<img src="images/banner.png" style="width: 100%;">

In [1]:
import os
os.environ["KERAS_BACKEND"] = "torch"

from matplotlib import rcParams
import matplotlib.pyplot as plt


# Some preambles for prettification
rcParams.update({'figure.figsize': (8, 6), 'axes.spines.top': False,
                 'axes.spines.right': False, 'axes.labelsize': 12,
                 'axes.titlesize': 12, 'axes.titleweight': 'bold',
                 'lines.linewidth': 1.5})

# Machine Translation using Sequential Models

## 1 Data Preparation

In [2]:
import keras

### Downloading the Dataset: English-to-Spanish Translation

In [3]:
import pathlib

In [4]:
zip_path = keras.utils.get_file(
    origin=(
        "http://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip"
    ),
    fname="spa-eng",
    extract=True,
)
text_path = pathlib.Path(zip_path) / "spa-eng" / "spa.txt"

In [19]:
with open(text_path) as f:
    lines = f.read().split("\n")[:-1]

# Append start and end tokens
text_pairs = []
for line in lines:
    english, spanish = line.split("\t")
    spanish = "[start] " + spanish + " [end]"
    text_pairs.append((english, spanish))

In [20]:
len(text_pairs)

118964

In [21]:
import numpy as np

In [31]:
np.random.seed(120)
text_pairs[np.random.randint(len(text_pairs))]

("I'm busy at the moment.", '[start] En este momento estoy ocupado. [end]')

### Data Segregation

In [32]:
np.random.seed(1337)
np.random.shuffle(text_pairs)

val_samples = int(0.15 * len(text_pairs))
train_samples = len(text_pairs) - 2 * val_samples

train_pairs = text_pairs[:train_samples]
val_pairs = text_pairs[train_samples : train_samples + val_samples]
test_pairs = text_pairs[train_samples + val_samples :]

In [33]:
train_pairs[0]

('Thanks for asking.', '[start] Gracias por preguntarme. [end]')

### Standardization

In [34]:
import string

In [35]:
strip_chars = string.punctuation + "¿"
strip_chars = strip_chars.replace("[", "")
strip_chars = strip_chars.replace("]", "")

In [36]:
import re

In [37]:
def custom_standardization(input_string):
    lowercase = tf.strings.lower(input_string)
    return tf.strings.regex_replace(
        lowercase, f"[{re.escape(strip_chars)}]", ""
    )

### Integer Encoding

In [38]:
from keras import layers
import tensorflow as tf

In [39]:
vocab_size = 15000
sequence_length = 20

# Integer Encoding
english_tokenizer = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length,
)
spanish_tokenizer = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length + 1,
    standardize=custom_standardization,
)
train_english_texts = [pair[0] for pair in train_pairs]
train_spanish_texts = [pair[1] for pair in train_pairs]
english_tokenizer.adapt(train_english_texts)
spanish_tokenizer.adapt(train_spanish_texts)

### Defining Model Features and Target

<img src="images/encoder-decoder.png" style="width: 50%;">

In [40]:
batch_size = 64


def format_dataset(eng, spa):
    eng = english_tokenizer(eng)
    spa = spanish_tokenizer(spa)
    features = {"english": eng, "spanish": spa[:, :-1]}
    labels = spa[:, 1:]
    sample_weights = labels != 0

    return features, labels, sample_weights


def make_dataset(pairs):
    eng_texts, spa_texts = zip(*pairs)
    eng_texts = list(eng_texts)
    spa_texts = list(spa_texts)
    dataset = tf.data.Dataset.from_tensor_slices((eng_texts, spa_texts))
    dataset = dataset.batch(batch_size)
    dataset = dataset.map(format_dataset, num_parallel_calls=4)

    return dataset.shuffle(2048).cache()

train_ds = make_dataset(train_pairs)
val_ds = make_dataset(val_pairs)

In [41]:
inputs, targets, sample_weights = next(iter(train_ds))
print(inputs["english"].shape)

(64, 20)


In [42]:
print(inputs["spanish"].shape)

(64, 20)


In [43]:
print(targets.shape)

(64, 20)


In [44]:
print(sample_weights.shape)

(64, 20)


## 2 Sequential Models: RNNs

### Model Architecture

In [45]:
embed_dim = 256
hidden_dim = 1024

source = keras.Input(shape=(None,), dtype="int32", name="english")

# Encoder
x = layers.Embedding(vocab_size, embed_dim, mask_zero=True)(source)
rnn_layer = layers.GRU(hidden_dim)
rnn_layer = layers.Bidirectional(rnn_layer, merge_mode="sum")
encoder_output = rnn_layer(x)

target = keras.Input(shape=(None,), dtype="int32", name="spanish")

# Decoder
x = layers.Embedding(vocab_size, embed_dim, mask_zero=True)(target)
rnn_layer = layers.GRU(hidden_dim, return_sequences=True)
x = rnn_layer(x, initial_state=encoder_output)
x = layers.Dropout(0.5)(x)
target_predictions = layers.Dense(vocab_size, activation="softmax")(x)

seq2seq_rnn = keras.Model([source, target], target_predictions)

In [46]:
seq2seq_rnn.summary(line_length=80)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)          ┃ Output Shape      ┃     Param # ┃ Connected to       ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━┩
│ english (InputLayer)  │ (None, None)      │           0 │ -                  │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ spanish (InputLayer)  │ (None, None)      │           0 │ -                  │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ embedding (Embedding) │ (None, None, 256) │   3,840,000 │ english[0][0]      │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ not_equal (NotEqual)  │ (None, None)      │           0 │ english[0][0]      │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ embedding_1           │ (None, None, 256) │   3,840,000 │ spanish[0][0]      │
│ (Embedding)           │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ bidirectional         │ (None, 1024)      │   7,876,608 │ embedding[0][0],   │
│ (Bidirectional)       │                   │             │ not_equal[0][0]    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ gru_1 (GRU)           │ (None, None,      │   3,938,304 │ embedding_1[0][0], │
│                       │ 1024)             │             │ bidirectional[0][… │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ dropout (Dropout)     │ (None, None,      │           0 │ gru_1[0][0]        │
│                       │ 1024)             │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ dense (Dense)         │ (None, None,      │  15,375,000 │ dropout[0][0]      │
│                       │ 15000)            │             │                    │
└───────────────────────┴───────────────────┴─────────────┴────────────────────┘

 Total params: 34,869,912 (133.02 MB)

 Trainable params: 34,869,912 (133.02 MB)

 Non-trainable params: 0 (0.00 B)

### Model Training

In [ ]:
seq2seq_rnn.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    weighted_metrics=["accuracy"],
)
seq2seq_rnn.fit(train_ds, epochs=15, validation_data=val_ds)

Runtime: 3 mins $\times$ 15 epochs = 45 mins

In [58]:
test_ds = make_dataset(test_pairs)

In [59]:
seq2seq_rnn.evaluate(test_ds)

279/279 ━━━━━━━━━━━━━━━━━━━━ 24s 84ms/step - accuracy: 0.6554 - loss: 1.9099


[1.9099243879318237, 0.6553820371627808]

### Model Deployment

In [49]:
spa_vocab = spanish_tokenizer.get_vocabulary()
spa_index_lookup = dict(zip(range(len(spa_vocab)), spa_vocab))

def rnn_generate_translation(input_sentence):
    tokenized_input_sentence = english_tokenizer([input_sentence])

    decoded_sentence = "[start]"
    for i in range(sequence_length):
        tokenized_target_sentence = spanish_tokenizer([decoded_sentence])
        inputs = [tokenized_input_sentence, tokenized_target_sentence]
        next_token_predictions = seq2seq_rnn.predict(inputs, verbose=0)
        sampled_token_index = np.argmax(next_token_predictions[0, i, :])
        sampled_token = spa_index_lookup[sampled_token_index]
        decoded_sentence += " " + sampled_token

        if sampled_token == "[end]":
            break

    return decoded_sentence

In [50]:
test_eng_texts = [pair[0] for pair in test_pairs]

for _ in range(5):
    input_sentence = random.choice(test_eng_texts)
    print("-")
    print(input_sentence)
    print(rnn_generate_translation(input_sentence))

-
It was so noisy in there.
[start] estaba tan caliente por ahí [end]
-
This button is loose.
[start] este botón está suelto [end]
-
I almost caught the fish.
[start] casi me pez pescado el pescado [end]
-
Visitors may not feed the animals.
[start] no se puede dar de comer a los animales [end]
-
Are both of you ready to go?
[start] ambos están a punto de listos [end]


## 2 Transformer Architecture

### Transformer Encoder

In [51]:
class TransformerEncoder(keras.Layer):
    def __init__(self, hidden_dim, intermediate_dim, num_heads):
        super().__init__()
        key_dim = hidden_dim // num_heads
        self.self_attention = layers.MultiHeadAttention(num_heads, key_dim)
        self.self_attention_layernorm = layers.LayerNormalization()
        self.feed_forward_1 = layers.Dense(intermediate_dim, activation="relu")
        self.feed_forward_2 = layers.Dense(hidden_dim)
        self.feed_forward_layernorm = layers.LayerNormalization()

    def call(self, source, source_mask):
        residual = x = source
        mask = source_mask[:, None, :]

        x = self.self_attention(query=x, key=x, value=x, attention_mask=mask)
        x = x + residual
        x = self.self_attention_layernorm(x)
        residual = x
        x = self.feed_forward_1(x)
        x = self.feed_forward_2(x)
        x = x + residual
        x = self.feed_forward_layernorm(x)

        return x

### Transformer Decoder

In [52]:
class TransformerDecoder(keras.Layer):
    def __init__(self, hidden_dim, intermediate_dim, num_heads):
        super().__init__()
        key_dim = hidden_dim // num_heads
        self.self_attention = layers.MultiHeadAttention(num_heads, key_dim)
        self.self_attention_layernorm = layers.LayerNormalization()
        self.cross_attention = layers.MultiHeadAttention(num_heads, key_dim)
        self.cross_attention_layernorm = layers.LayerNormalization()
        self.feed_forward_1 = layers.Dense(intermediate_dim, activation="relu")
        self.feed_forward_2 = layers.Dense(hidden_dim)
        self.feed_forward_layernorm = layers.LayerNormalization()

    def call(self, target, source, source_mask):
        residual = x = target

        x = self.self_attention(query=x, key=x, value=x, use_causal_mask=True)
        x = x + residual
        x = self.self_attention_layernorm(x)
        residual = x

        mask = source_mask[:, None, :]
        x = self.cross_attention(
            query=x, key=source, value=source, attention_mask=mask
        )
        x = x + residual
        x = self.cross_attention_layernorm(x)
        residual = x
        x = self.feed_forward_1(x)
        x = self.feed_forward_2(x)
        x = x + residual
        x = self.feed_forward_layernorm(x)
        return x

### Sequential Modeling using Transformer

In [53]:
hidden_dim = 256
intermediate_dim = 2048
num_heads = 8

source = keras.Input(shape=(None,), dtype="int32", name="english")
x = layers.Embedding(vocab_size, hidden_dim)(source)
encoder_output = TransformerEncoder(hidden_dim, intermediate_dim, num_heads)(
    source=x,
    source_mask=source != 0,
)

target = keras.Input(shape=(None,), dtype="int32", name="spanish")
x = layers.Embedding(vocab_size, hidden_dim)(target)
x = TransformerDecoder(hidden_dim, intermediate_dim, num_heads)(
    target=x,
    source=encoder_output,
    source_mask=source != 0,
)
x = layers.Dropout(0.5)(x)
target_predictions = layers.Dense(vocab_size, activation="softmax")(x)

transformer = keras.Model([source, target], target_predictions)

In [54]:
transformer.summary(line_length=80)

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)          ┃ Output Shape      ┃     Param # ┃ Connected to       ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━┩
│ english (InputLayer)  │ (None, None)      │           0 │ -                  │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ embedding_2           │ (None, None, 256) │   3,840,000 │ english[0][0]      │
│ (Embedding)           │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ not_equal_2           │ (None, None)      │           0 │ english[0][0]      │
│ (NotEqual)            │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ spanish (InputLayer)  │ (None, None)      │           0 │ -                  │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ transformer_encoder   │ (None, None, 256) │   1,315,072 │ embedding_2[0][0], │
│ (TransformerEncoder)  │                   │             │ not_equal_2[0][0]  │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ not_equal_3           │ (None, None)      │           0 │ english[0][0]      │
│ (NotEqual)            │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ embedding_3           │ (None, None, 256) │   3,840,000 │ spanish[0][0]      │
│ (Embedding)           │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ transformer_decoder   │ (None, None, 256) │   1,578,752 │ transformer_encod… │
│ (TransformerDecoder)  │                   │             │ not_equal_3[0][0], │
│                       │                   │             │ embedding_3[0][0]  │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ dropout_4 (Dropout)   │ (None, None, 256) │           0 │ transformer_decod… │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ dense_5 (Dense)       │ (None, None,      │   3,855,000 │ dropout_4[0][0]    │
│                       │ 15000)            │             │                    │
└───────────────────────┴───────────────────┴─────────────┴────────────────────┘

 Total params: 14,428,824 (55.04 MB)

 Trainable params: 14,428,824 (55.04 MB)

 Non-trainable params: 0 (0.00 B)

In [55]:
transformer.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    weighted_metrics=["accuracy"],
)
transformer.fit(train_ds, epochs=15, validation_data=val_ds)

Epoch 1/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 67s 51ms/step - accuracy: 0.3549 - loss: 1.5017 - val_accuracy: 0.4927 - val_loss: 1.0447
Epoch 2/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 65s 50ms/step - accuracy: 0.5291 - loss: 0.9809 - val_accuracy: 0.5713 - val_loss: 0.8313
Epoch 3/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 62s 48ms/step - accuracy: 0.6015 - loss: 0.7667 - val_accuracy: 0.5996 - val_loss: 0.7514
Epoch 4/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 66s 51ms/step - accuracy: 0.6461 - loss: 0.6389 - val_accuracy: 0.6140 - val_loss: 0.7200
Epoch 5/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 67s 52ms/step - accuracy: 0.6804 - loss: 0.5496 - val_accuracy: 0.6241 - val_loss: 0.7165
Epoch 6/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 68s 52ms/step - accuracy: 0.7072 - loss: 0.4837 - val_accuracy: 0.6276 - val_loss: 0.7125
Epoch 7/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 67s 52ms/step - accuracy: 0.7286 - loss: 0.4344 - val_accuracy: 0.6308 - val_loss: 0.7187
Epoch 8/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 70s 54ms/step - accuracy: 0.7786 -

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



Runtime: 70 seconds $\times$ 15 = 17.5 minutes

In [60]:
transformer.evaluate(test_ds)

279/279 ━━━━━━━━━━━━━━━━━━━━ 7s 24ms/step - accuracy: 0.6359 - loss: 0.8240


[0.823963463306427, 0.6359102129936218]

### Transformer with Positional Embedding

In [61]:
from keras import ops

In [62]:
class PositionalEmbedding(keras.Layer):
    def __init__(self, sequence_length, input_dim, output_dim):
        super().__init__()
        self.token_embeddings = layers.Embedding(input_dim, output_dim)
        self.position_embeddings = layers.Embedding(sequence_length, output_dim)

    def call(self, inputs):
        positions = ops.cumsum(ops.ones_like(inputs), axis=-1) - 1
        embedded_tokens = self.token_embeddings(inputs)
        embedded_positions = self.position_embeddings(positions)

        return embedded_tokens + embedded_positions

In [64]:
hidden_dim = 256
intermediate_dim = 2056
num_heads = 8

source = keras.Input(shape=(None,), dtype="int32", name="english")
x = PositionalEmbedding(sequence_length, vocab_size, hidden_dim)(source)
encoder_output = TransformerEncoder(hidden_dim, intermediate_dim, num_heads)(
    source=x,
    source_mask=source != 0,
)

target = keras.Input(shape=(None,), dtype="int32", name="spanish")

x = PositionalEmbedding(sequence_length, vocab_size, hidden_dim)(target)
x = TransformerDecoder(hidden_dim, intermediate_dim, num_heads)(
    target=x,
    source=encoder_output,
    source_mask=source != 0,
)
x = layers.Dropout(0.5)(x)
target_predictions = layers.Dense(vocab_size, activation="softmax")(x)

transformer_pos_embed = keras.Model([source, target], target_predictions)

In [65]:
transformer_pos_embed.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    weighted_metrics=["accuracy"],
)
transformer_pos_embed.fit(train_ds, epochs=30, validation_data=val_ds)

Epoch 1/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 71s 55ms/step - accuracy: 0.4020 - loss: 1.3938 - val_accuracy: 0.5626 - val_loss: 0.9080
Epoch 2/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 71s 55ms/step - accuracy: 0.5956 - loss: 0.8442 - val_accuracy: 0.6378 - val_loss: 0.7057
Epoch 3/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 71s 55ms/step - accuracy: 0.6642 - loss: 0.6446 - val_accuracy: 0.6666 - val_loss: 0.6298
Epoch 4/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 71s 55ms/step - accuracy: 0.7029 - loss: 0.5320 - val_accuracy: 0.6793 - val_loss: 0.6058
Epoch 5/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 66s 51ms/step - accuracy: 0.7304 - loss: 0.4586 - val_accuracy: 0.6819 - val_loss: 0.5999
Epoch 6/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 63s 48ms/step - accuracy: 0.7532 - loss: 0.4038 - val_accuracy: 0.6828 - val_loss: 0.6024
Epoch 7/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 72s 55ms/step - accuracy: 0.7718 - loss: 0.3615 - val_accuracy: 0.6887 - val_loss: 0.6024
Epoch 8/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 73s 56ms/step - accuracy: 0.7875 -

Runtime: 70 seconds $\times$ 30 = 35 minutes

In [67]:
transformer_pos_embed.evaluate(test_ds)

279/279 ━━━━━━━━━━━━━━━━━━━━ 7s 25ms/step - accuracy: 0.6898 - loss: 0.8081


[0.8081309795379639, 0.6898049712181091]

### Model Deployment

In [68]:
def transformer_generate_translation(input_sentence, model):
    tokenized_input_sentence = english_tokenizer([input_sentence])
    decoded_sentence = "[start]"
    for i in range(sequence_length):
        tokenized_target_sentence = spanish_tokenizer([decoded_sentence])
        tokenized_target_sentence = tokenized_target_sentence[:, :-1]
        inputs = [tokenized_input_sentence, tokenized_target_sentence]
        next_token_predictions = model.predict(inputs, verbose=0)
        sampled_token_index = np.argmax(next_token_predictions[0, i, :])
        sampled_token = spa_index_lookup[sampled_token_index]
        decoded_sentence += " " + sampled_token
        if sampled_token == "[end]":
            break

    return decoded_sentence

In [72]:
test_eng_texts = [pair[0] for pair in test_pairs]
for _ in range(5):
    input_sentence = random.choice(test_eng_texts)
    print("-")
    print(input_sentence)
    print(transformer_generate_translation(input_sentence, transformer_pos_embed))

-
Russia is the largest country in the world.
[start] rusia es el campo más grande del mundo [end]
-
What is your number?
[start] cuál es tu número [end]
-
Tom is a social drinker.
[start] tom es un [UNK] más grande [end]
-
I called him up on the phone.
[start] lo llamé al teléfono [end]
-
Tom begged me to come.
[start] tom me [UNK] [end]


<img src="images/banner-down.png" style="width: 100%;">